# Demo and Test of Parallel implimentation of GSTATSIM interpolation function
This notebook provides a demo of how to use each parallelized interpolation function. Additionally it is used to validate the parallelized implimentation by comparing it to the output of the original serialized implimentation.

## Initialize data and simulation grid

In [2]:
import sys
sys.path.append("../")

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import LightSource
from sklearn.preprocessing import QuantileTransformer
import skgstat as skg
from skgstat import models
import gstatsim as gs
import parallel

In [4]:
# load data
df_bed = pd.read_csv('../demos/data/greenland_test_data.csv')
# remove erroneously high values due to bad bed picks
df_bed = df_bed[df_bed["Bed"] <= 700]  

In [5]:
# grid data to 100 m resolution and remove coordinates with NaNs
res = 1000
df_grid, grid_matrix, rows, cols = gs.Gridding.grid_data(df_bed, 'X', 'Y', 'Bed', res)
df_grid = df_grid[df_grid["Z"].isnull() == False]
df_grid = df_grid.rename(columns = {"Z": "Bed"})

# normal score transformation
data = df_grid['Bed'].values.reshape(-1,1)
nst_trans = QuantileTransformer(n_quantiles=500, output_distribution="normal").fit(data)
df_grid['Nbed'] = nst_trans.transform(data) 

# compute experimental (isotropic) variogram
coords = df_grid[['X','Y']].values
values = df_grid['Nbed']

maxlag = 50000             # maximum range distance
n_lags = 70                # num of bins

V1 = skg.Variogram(coords, values, bin_func='even', n_lags=n_lags, 
                   maxlag=maxlag, normalize=False)

# use exponential variogram model
V1.model = 'exponential'
V1.parameters

[31852.813365632115, 0.7027482242025527, 0]

In [6]:
# define coordinate grid
xmin = np.min(df_grid['X']); xmax = np.max(df_grid['X'])     # min and max x values
ymin = np.min(df_grid['Y']); ymax = np.max(df_grid['Y'])     # min and max y values

# initialize grid
Pred_grid_xy = gs.Gridding.prediction_grid(xmin, xmax, ymin, ymax, res)

In [7]:
# set variogram parameters
azimuth = 0
nugget = V1.parameters[2]

# the major and minor ranges are the same in this example because it is isotropic
major_range = V1.parameters[0]
minor_range = V1.parameters[0]
sill = V1.parameters[1]
vtype = 'Exponential'

# save variogram parameters as a list
vario = [azimuth, nugget, major_range, minor_range, sill, vtype]


k = 50         # number of neighboring data points used to estimate a given point
rad = 50000     # 50 km search radius

## Simple and Ordinary Kriging

In [8]:
est_SK, var_SK = parallel.skrige_interp(Pred_grid_xy, df_grid, 'X', 'Y', 'Nbed', k, vario, rad, 10)

In [9]:
GT_skrig = np.loadtxt('GT/skrig.csv', delimiter=',')
GT_est_SK = GT_skrig[:,0]
GT_var_SK = GT_skrig[:,1]

In [17]:
print(f'SSE elevation estimates: {np.sum(np.square(GT_est_SK - est_SK))}')
print(f'SSE elevation variance: {np.sum(np.square(GT_var_SK - var_SK))}')

SSE elevation estimates: 0.0
SSE elevation variance: 0.0


In [13]:
est_OK, var_OK = parallel.okrige_interp(Pred_grid_xy, df_grid, 'X', 'Y', 'Nbed', k, vario, rad, 10)

In [14]:
GT_okrig = np.loadtxt('GT/okrig.csv', delimiter=',')
GT_est_OK = GT_okrig[:,0]
GT_var_OK = GT_okrig[:,1]

In [18]:
print(f'SSE elevation estimates: {np.sum(np.square(GT_est_OK - est_OK))}')
print(f'SSE elevation variance: {np.sum(np.square(GT_var_OK - var_OK))}')

SSE elevation estimates: 0.0
SSE elevation variance: 0.0


## SGS with Simple and Ordinary Kriging
Since SGS is a stochastic process, instead of comparing a single realization we will compare a small ensemble of realizations